## Script to parse through the jsons of the politicians and store them in a local postgres db

In [1]:
from __future__ import annotations

import os
import re
import json
import hashlib
from dataclasses import dataclass
from datetime import date
from typing import Iterator, List, Optional, TypedDict, NewType, Literal, Any

import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
from collections import Counter

from file_handling.file_read_writer import write_json, read_json
from params.paths import ROOT_DIR


LOWER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_shugiin', 'repr_list')
LOWER_HOUSE_DATA_HISTORICAL_TMP = os.path.join(LOWER_HOUSE_DATA_DIR, 'historical')
LOWER_HOUSE_DATA_CURRENT = os.path.join(LOWER_HOUSE_DATA_DIR, 'current')
UPPER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_sangiin', 'repr_list')
UPPER_HOUSE_DATA_CURRENT = os.path.join(UPPER_HOUSE_DATA_DIR, 'current')
UPPER_HOUSE_DATA_HISTORICAL_TMP = os.path.join(UPPER_HOUSE_DATA_DIR, 'historical')
os.makedirs(LOWER_HOUSE_DATA_HISTORICAL_TMP, exist_ok=True)
os.makedirs(UPPER_HOUSE_DATA_HISTORICAL_TMP, exist_ok=True)



In [2]:
PersonId = NewType("PersonId", int)

class RawElection(TypedDict):
    year: str          # e.g., "1952年"
    month: str         # e.g., "10月"
    day: str           # e.g., "1日"
    election_name: str
    district: str
    party: str
    result: str        # e.g., "当選"
    election_freq: str # e.g., "（1回目）"

class RawPersonFile(TypedDict):
    name_kanji: str
    name_kana: str
    election_data: List[RawElection]

@dataclass(frozen=True)
class Person:
    name_kanji: str
    name_kana: str
    election_signature: str

@dataclass(frozen=True)
class ElectionResult:
    person_id: PersonId
    election_date: date
    election_name: str
    district: str
    party: str
    result: str
    election_number: Optional[int]  # parsed from "（1回目）"
    raw_json: dict[str, Any]


In [3]:
DDL: list[str] = [
    """CREATE TABLE IF NOT EXISTS person (
        person_id BIGSERIAL PRIMARY KEY,
        name_kanji TEXT NOT NULL,
        name_kana  TEXT,
        election_signature TEXT,
        CONSTRAINT uq_person UNIQUE (name_kanji, name_kana, election_signature)
    );""",
    """CREATE TABLE IF NOT EXISTS election_result (
        id BIGSERIAL PRIMARY KEY,
        person_id BIGINT NOT NULL REFERENCES person(person_id),
        election_date DATE NOT NULL,
        election_name TEXT,
        district TEXT,
        party TEXT,
        result TEXT,
        election_number INT,
        raw_json JSONB,
        CONSTRAINT uq_election UNIQUE (person_id, election_date, election_name, district, party)
    );"""
]


In [4]:
DIGITS = re.compile(r"\d+")

def _parse_int(s: str) -> Optional[int]:
    m = DIGITS.search(s)
    return int(m.group(0)) if m else None

def parse_election_date(y: str, m: str, d: str) -> date:
    yy = _parse_int(y) or 1
    mm = _parse_int(m) or 1
    dd = _parse_int(d) or 1
    return date(yy, mm, dd)

def parse_election_number(freq: Optional[str]) -> Optional[int]:
    # "（1回目）" -> 1
    if freq is None or freq == "":
        return None
    return _parse_int(freq)

def generate_election_signature(raw: RawPersonFile) -> str:
    # Deterministic: hash sorted key/value pairs of the FIRST election entry
    # (Adjust to include more entries if you want a stronger signature)
    if not raw["election_data"]:
        base = f"{raw['name_kanji']}|{raw['name_kana']}|no_elections"
    else:
        first = raw["election_data"][0]
        parts = [f"{k}:{first[k]}" for k in sorted(first.keys())]
        base = f"{raw['name_kanji']}|{raw['name_kana']}|" + "|".join(parts)
    return hashlib.sha256(base.encode("utf-8")).hexdigest()

In [5]:
from utils.string_process import clean_repr_name

# ---------- IO over your historical dirs ----------
def iterate_over_historical_data(house_historical_dir: str) -> Iterator[RawPersonFile]:
    for file in os.listdir(house_historical_dir):
        if not file.endswith(".json"):
            continue
        data = read_json(os.path.join(house_historical_dir, file))  # type: ignore[name-defined]
        # Lightweight runtime shape check; keep it cheap
        if not isinstance(data, dict):
            continue
        if not data.get("name_kana"):
            continue
        yield data  # type: ignore[typeddict-item]

def iterate_over_current_data(json_path: str) -> Iterator[RawPersonFile]:
    data = read_json(json_path)
    for repr in data['reprs']:
        yield clean_repr_name(repr['name']), clean_repr_name(repr['yomikata']), repr

# ---------- DB ops ----------
def run_ddl(cur: psycopg2.extensions.cursor) -> None:
    for stmt in DDL:
        cur.execute(stmt)

def insert_person(cur: psycopg2.extensions.cursor, p: Person) -> PersonId:
    cur.execute(
        """
        INSERT INTO person (name_kanji, name_kana, election_signature)
        VALUES (%s, %s, %s)
        ON CONFLICT (name_kanji, name_kana, election_signature) DO UPDATE
        SET name_kana = EXCLUDED.name_kana
        RETURNING person_id;
        """,
        (p.name_kanji, p.name_kana, p.election_signature),
    )
    pid = cur.fetchone()[0]
    return PersonId(pid)

def insert_elections_bulk(cur: psycopg2.extensions.cursor, rows: list[ElectionResult]) -> None:
    if not rows:
        return
    execute_values(
        cur,
        """
        INSERT INTO election_result (
            person_id, election_date, election_name, district, party, result, election_number, raw_json
        )
        VALUES %s
        ON CONFLICT DO NOTHING;
        """,
        [
            (
                int(r.person_id),
                r.election_date,
                r.election_name,
                r.district,
                r.party,
                r.result,
                r.election_number,
                json.dumps(r.raw_json)
            )
            for r in rows
        ],
    )

def upsert_person_and_elections(cur: psycopg2.extensions.cursor, raw: RawPersonFile) -> None:
    sig = generate_election_signature(raw)
    person = Person(
        name_kanji=raw["name_kanji"],
        name_kana=raw["name_kana"],
        election_signature=sig,
    )
    pid = insert_person(cur, person)

    results: list[ElectionResult] = []
    for e in raw["election_data"]:
        results.append(
            ElectionResult(
                person_id=pid,
                election_date=parse_election_date(e["year"], e["month"], e["day"]),
                election_name=e["election_name"],
                district=e["district"],
                party=e["party"],
                result=e["result"],
                election_number=parse_election_number(e.get("election_freq")),
                raw_json=e,
            )
        )
    insert_elections_bulk(cur, results)

## Checking whether all the duplicates in the db aare actual different people

In [ ]:
from dotenv import load_dotenv
import psycopg2
from collections import Counter
import os
from typing import List, Dict, Tuple, Any, Optional, Set

from api_requests.prompter import DeepResearchGemini

load_dotenv()

prompter = DeepResearchGemini(model_name="gemini-3.1-pro-preview")

duplication_check_sys_prompt = """
You are a helpful assistant that checks whether the politicians are the same person.
You will be given a list of politicians and their election data.
You will need to check whether they are the same person.
If they are the same person, you just respond "SAMEPERSON" We will pass the merging process to the next assistant.
If they are not the same person, you will respond "DIFFERENTPERSON". We will assume we do not have to merge the records.
"""

merging_sys_prompt = """
You are a helpful assistant that merges the election data of the politicians.
I will give you entries from my database that have the same name but different ids. These records were found to be duplicates by the previous assistant and need to be merged into a single, new record. Make sure you do research to verify the entries and create a new record that is accurate. You must supply your reply in the following format. Make sure you do not include any extra characters or comments or code blocks such as ```json.

{
	"name_kanji": "名前",
	"name_kana": "名前のふりがな",
	"years": [
	"当選年-当選月-当選日",
	"当選年-当選月-当選日",
	...
	],
	"election_data": [
		{
			"year": "当選年",
			"month": "当選月",
			"day": "当選日",
			"election_name": "当選回次",
			"district": "選挙区",
			"party": "政党",
			"result": "当選",
			"election_freq": "（1回目）"
		},
		...
	]
	}

	返答フォーマットの例：
	{
		"name_kanji": "七条明",
		"name_kana": "しちじょうあきら",
		"years": [
			"1993-07-18",
			...
		],
		"election_data": [
			{
				"year": "1993年",
				"month": "7月",
				"day": "18日",
				"election_name": "第40回衆議院議員総選挙",
				"district": "徳島全県区",
				"party": "自由民主党",
				"result": "当選",
				"election_freq": "（1回目）"
			},
			...
		]
	}
"""

create_entry_sys_prompt = """
You are a helpful assistant that creates an entry for a politician in my database.
I will give you a politician's name and you will need to create an entry for them in my database. For the election history, make sure to only include federal elections.
Your response should be in the following format. Make sure you do not include any extra characters or comments or code blocks such as ```json.
{
	"name_kanji": "名前",
	"name_kana": "名前のふりがな",
	"years": [
	"当選年-当選月-当選日",
	"当選年-当選月-当選日",
	...
	],
	"election_data": [
		{
			"year": "当選年",
			"month": "当選月",
			"day": "当選日",
			"election_name": "当選回次",
			"district": "選挙区",
			"party": "政党",
			"result": "当選",
			"election_freq": "（1回目）"
		},
		...
	]
}

返答フォーマットの例：
{
	"name_kanji": "七条明",
	"name_kana": "しちじょうあきら",
	"years": [
		"1993-07-18",
		...
	],
	"election_data": [
		{
			"year": "1993年",
			"month": "7月",
			"day": "18日",
			"election_name": "第40回衆議院議員総選挙",
			"district": "徳島全県区",
			"party": "自由民主党",
			"result": "当選",
			"election_freq": "（1回目）"
		},
		...
	]
}
"""

refresh_election_history_sys_prompt = """
You maintain a Japanese Diet politicians database. A row already exists with the same name_kanji or
name_kana as the incoming record. Use web search to verify facts and return the most accurate, current
**national (federal) Diet election history only** (衆議院・参議院の国政選挙).

Reply with a single JSON object only. No markdown fences or commentary. Same shape as when creating a new entry:
{
  "name_kanji": "名前",
  "name_kana": "ふりがな",
  "years": ["YYYY-MM-DD", ...],
  "election_data": [
    {
      "year": "2024年",
      "month": "10月",
      "day": "27日",
      "election_name": "第50回衆議院議員総選挙",
      "district": "…",
      "party": "…",
      "result": "当選",
      "election_freq": "（1回目）"
    }
  ]
}

List every federal election you can verify for this person (complete history). The application will **only
insert rows that are not already in the database**; the user message includes existing election rows per person_id.

If multiple person_id candidates appear (same name or reading, different rows), you MUST include
"target_person_id": <integer> for the one row that matches this politician (use OTHER INFO, districts,
dates, and search). If exactly one candidate is listed, you may omit target_person_id.
"""


def _collect_existing_person_candidates(
	cur: psycopg2.extensions.cursor, name: str, hiragana: Optional[str]
) -> List[Tuple[Any, ...]]:
	seen: Dict[int, Tuple[Any, ...]] = {}
	cur.execute(
		"SELECT person_id, name_kanji, name_kana, election_signature FROM person WHERE name_kanji = %s",
		(name,),
	)
	for row in cur.fetchall():
		seen[row[0]] = row
	if hiragana:
		cur.execute(
			"SELECT person_id, name_kanji, name_kana, election_signature FROM person WHERE name_kana = %s",
			(hiragana,),
		)
		for row in cur.fetchall():
			seen[row[0]] = row
	return list(seen.values())


def _election_rows_for_person(cur: psycopg2.extensions.cursor, person_id: int) -> List[Tuple[Any, ...]]:
	cur.execute(
		"""
		SELECT election_date, election_name, district, party, result, election_number, raw_json
		FROM election_result
		WHERE person_id = %s
		ORDER BY election_date
		""",
		(person_id,),
	)
	return list(cur.fetchall())


def _existing_election_keys(cur: psycopg2.extensions.cursor, person_id: int) -> Set[Tuple[Any, ...]]:
	cur.execute(
		"""
		SELECT election_date, election_name, district, party
		FROM election_result
		WHERE person_id = %s
		""",
		(person_id,),
	)
	return {tuple(row) for row in cur.fetchall()}


def compose_refresh_election_history_prompt(
	candidates: List[Tuple[Any, ...]],
	election_rows_by_pid: Dict[int, List[Tuple[Any, ...]]],
	name: str,
	hiragana: Optional[str],
	other_info: Optional[str],
) -> str:
	parts: List[str] = []
	parts.append(f"Incoming record — name_kanji: {name}")
	if hiragana:
		parts.append(f"Incoming name_kana: {hiragana}")
	if other_info:
		parts.append(f"OTHER INFO:\n{other_info}")
	parts.append("\n--- Existing DB person rows that share this name_kanji or name_kana (with stored elections) ---")
	for pid, nk, nn, sig in candidates:
		sig_short = (sig[:20] + "…") if sig and len(sig) > 20 else (sig or "")
		parts.append(f"\nperson_id={pid} | name_kanji={nk} | name_kana={nn} | election_signature_prefix={sig_short}")
		rows = election_rows_by_pid.get(pid, [])
		if not rows:
			parts.append("  (no election_result rows)")
		else:
			for r in rows:
				parts.append(
					f"  date={r[0]} | election={r[1]} | district={r[2]} | party={r[3]} | result={r[4]} | n={r[5]}"
				)
	parts.append(
		"\nResearch and return refreshed federal election JSON per the system prompt. JSON only."
	)
	return "\n".join(parts)



def get_duplicate_politicians(cur: psycopg2.extensions.cursor):
    cur.execute("""
    SELECT p.*
    FROM public.person p
    JOIN (
        SELECT name_kanji
        FROM public.person
        GROUP BY name_kanji
        HAVING COUNT(*) > 1
    ) dup
    ON p.name_kanji = dup.name_kanji
    ORDER BY p.name_kanji;
    """)
    return cur.fetchall()

def create_name2ids_dict(duplicates:List):
    name2ids = {}
    for id, name, *_ in duplicates:
        if name not in name2ids:
            name2ids[name] = []
        name2ids[name].append(id)
    return name2ids

def compose_duplicate_check_prompt(name: str, id2elections:Dict[int, List[Tuple[str, Any]]]):
	prompt = f"""
	We have a DB with the following entries for politicians with the same name but different ids:	
	POLITICIAN NAME: {name}
	"""

	compose_election_record_str = lambda id, elections: f"""
	POLITICIAN ID: {id}
	ELECTION RECORD:
	{"\n".join([f"{'---'.join([str(e) for e in election[2:]])}" for election in elections])}
	"""
	for id, elections in id2elections.items():
		prompt += compose_election_record_str(id, elections)

	prompt += "\n\nCheck whether they are the same person by following the response format given in system prompt. Make sure you do some research on the internet if you need to."
	

	return prompt

def compose_merging_prompt(name:str, id2elections:Dict[int, List[Tuple[str, Any]]]):
	prompt = f"""
	POLITICIAN NAME: {name}
	"""

	compose_election_record_str = lambda id, elections: f"""
	POLITICIAN ID: {id}
	ELECTION RECORD:
	{"\n".join([f"{'---'.join([str(e) for e in election[2:]])}" for election in elections])}
	"""	
	for id, elections in id2elections.items():
		prompt += compose_election_record_str(id, elections)

	prompt += "\n\nMerge the election data of the politicians while verifying the data by doing some research on the internet."
	prompt += "\n\nMake sure you do not include any extra characters or comments or code blocks such as ```json."

	return prompt

def compose_create_entry_prompt(name:str, other_info:str, cur:psycopg2.extensions.cursor)->str:
	prompt = f"""
	Create an entry for the following politician in my database. Make sure to follow the format given in the system prompt.
	POLITICIAN NAME: {name}
	{f"OTHER INFO ABOUT THE POLITICIAN: {other_info}" if other_info else ""}

	"""

	return prompt

def merge_and_update_db(name:str, id2elections:Dict[int, List[Tuple[str, Any]]], cur:psycopg2.extensions.cursor)->None:
	count = 0
	while True:
		try:
			prompt = compose_merging_prompt(name, id2elections)
			response, _,_ = prompter.prompt(prompt, merging_sys_prompt)
			response = response.replace("```json", "").replace("```", "")
			response = json.loads(response)
			# resume = input(f"""Do you want to resume the merging process? (y/n)\nNAME OF REPRESENTATIVE: {name}\nNEW ELECTION DATA: {response}""")
			# if resume == "n":
			# 	raise Exception("User decided to stop the merging process.")
			new_person = Person(
				name_kanji=response["name_kanji"],
				name_kana=response["name_kana"],
				election_signature=generate_election_signature(response)
			)
			pid = insert_person(cur, new_person)
			print(f"Inserted new person with id: {pid}")

			new_results: list[ElectionResult] = []
			for e in response["election_data"]:
				new_results.append(
					ElectionResult(
						person_id=pid,
						election_date=parse_election_date(e["year"], e["month"], e["day"]),
						election_name=e["election_name"],
						district=e["district"],
						party=e["party"],
						result=e["result"],
						election_number=parse_election_number(e.get("election_freq")),
						raw_json=e
					)
				)
			insert_elections_bulk(cur, new_results)
			
			print(f"Deleted {len(id2elections)} duplicates from person table.")
			cur.execute(f"""
			DELETE FROM public.election_result WHERE person_id IN ({",".join([str(id) for id in id2elections.keys()])})
			""")
			print(f"Merged {name} with {len(id2elections)} duplicates.")
			cur.execute(f"""
			DELETE FROM public.person WHERE person_id IN ({",".join([str(id) for id in id2elections.keys()])})
			""")
			print(f"Deleted {len(id2elections)} duplicates from election_result table.")
			break
		except Exception as e:
			raise e

def check_and_merge_duplicates(name:str, ids:List[int], cur:psycopg2.extensions.cursor)->None:

	id2elections = {}
	
	# retrieve the election data for each id
	for id in ids:
		cur.execute(f"""
		SELECT er.* FROM public.election_result er WHERE er.person_id = {id}
		""")
		id2elections[id] = cur.fetchall()

	prompt = compose_duplicate_check_prompt(name, id2elections)
	count = 0
	while True:
		response, _,_ = prompter.prompt(prompt, duplication_check_sys_prompt)
		response = response.strip()
		if response == "SAMEPERSON":
			merge_and_update_db(name, id2elections, cur)
			break
		elif response == "DIFFERENTPERSON":
			break
		count += 1
		if count > 3:
			break

def create_entries_for_politician(name:str, hiragana:str=None,other_info:str=None,cur:psycopg2.extensions.cursor=None)->None:
	candidates = _collect_existing_person_candidates(cur, name, hiragana)
	if candidates:
		election_rows_by_pid: Dict[int, List[Tuple[Any, ...]]] = {}
		for row in candidates:
			pid = row[0]
			election_rows_by_pid[pid] = _election_rows_for_person(cur, pid)

		print("=== Existing DB politician(s) matching name_kanji or name_kana ===")
		for pid, nk, nn, sig in candidates:
			sig_short = (sig[:20] + "…") if sig and len(sig) > 20 else (sig or "")
			print(f"\n--- person_id={pid} | {nk} | {nn} | signature_prefix={sig_short} ---")
			rows = election_rows_by_pid.get(pid, [])
			if not rows:
				print("  (no election_result rows)")
			else:
				for r in rows:
					print(f"  {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]}")

		print("\n=== OTHER INFO for incoming record ===")
		print(other_info if other_info else "(none)")

		uprompt = compose_refresh_election_history_prompt(
			candidates, election_rows_by_pid, name, hiragana, other_info
		)
		count = 0
		response: Optional[Dict[str, Any]] = None
		while count <= 3:
			try:
				raw = prompter.prompt(uprompt, refresh_election_history_sys_prompt)
				raw = raw.strip().replace("```json", "").replace("```", "").strip()
				response = json.loads(raw)
				break
			except Exception:
				count += 1
				if count > 3:
					raise RuntimeError("Could not parse LLM refresh response as JSON after retries") from None

		assert response is not None
		cand_ids = [c[0] for c in candidates]
		if len(cand_ids) == 1:
			pid = PersonId(int(cand_ids[0]))
		else:
			tid = response.get("target_person_id")
			if tid is None:
				raise ValueError(
					f"Multiple person rows match name/kana; JSON must include target_person_id. Candidates: {cand_ids}"
				)
			tid = int(tid)
			if tid not in cand_ids:
				raise ValueError(f"LLM target_person_id {tid} not in candidates {cand_ids}")
			pid = PersonId(tid)

		existing_keys = _existing_election_keys(cur, int(pid))
		election_data = response.get("election_data") or []
		to_insert: list[ElectionResult] = []
		for e in election_data:
			ed = parse_election_date(e["year"], e["month"], e["day"])
			key = (ed, e["election_name"], e["district"], e["party"])
			if key in existing_keys:
				continue
			to_insert.append(
				ElectionResult(
					person_id=pid,
					election_date=ed,
					election_name=e["election_name"],
					district=e["district"],
					party=e["party"],
					result=e["result"],
					election_number=parse_election_number(e.get("election_freq")),
					raw_json=e,
				)
			)
		insert_elections_bulk(cur, to_insert)
		skipped = len(election_data) - len(to_insert)
		print(
			f"Refreshed person_id={pid}: inserted {len(to_insert)} new election row(s); "
			f"{skipped} already in DB (skipped)."
		)
		return

	count = 0
	while True:
		try:
			prompt = compose_create_entry_prompt(name, other_info, cur)
			response_text = prompter.prompt(prompt, create_entry_sys_prompt)
			response_text = response_text.replace("```json", "").replace("```", "")
			response = json.loads(response_text)
			new_person = Person(
				name_kanji=response["name_kanji"],
				name_kana=response["name_kana"],
				election_signature=generate_election_signature(response),
			)
			pid = insert_person(cur, new_person)
			print(f"Inserted new person with id: {pid}")
			new_results: list[ElectionResult] = []
			for e in response["election_data"]:
				new_results.append(
					ElectionResult(
						person_id=pid,
						election_date=parse_election_date(e["year"], e["month"], e["day"]),
						election_name=e["election_name"],
						district=e["district"],
						party=e["party"],
						result=e["result"],
						election_number=parse_election_number(e.get("election_freq")),
						raw_json=e,
					)
				)
			insert_elections_bulk(cur, new_results)
			break
		except Exception as e:
			count += 1
			if count > 3:
				raise e


In [16]:
from params.paths import DATA_DIR

skip_until = "にしだしょうじ"
conn = None
try:
	conn = psycopg2.connect(
		dbname="kokkaidoc",
		user="postgres",
		password=os.getenv("PSQL_DATABASE_PASSWORD"),
		host="localhost",
		port="5432",
	)
	print("Connected.")

	with conn.cursor() as cur:
		add_path = os.path.join(DATA_DIR, "data_shugiin", "repr_list", "20260315_repr_list.json")
		# Skip until name_kanji OR name_kana equals skip_until; then process that row and all following. Use None or "" to start from the beginning.
		reached = not (skip_until or "").strip()
		for name, hiragana, whole_info in iterate_over_current_data(add_path):
			if not reached:
				if name == skip_until or hiragana == skip_until:
					reached = True
				else:
					print(f"Skipping {name} {hiragana}")
					continue
			try:
				create_entries_for_politician(name, hiragana, json.dumps(whole_info,ensure_ascii=False), cur)
				conn.commit()
			except Exception as e:
				print(f"Error processing {name} {hiragana}: {e}")
			
			
		raise Exception("Stop here")
		create_entries_for = ['百田尚樹']

		for name in create_entries_for:
			create_entries_for_politician(name, cur)
			conn.commit()
		

		DEDUPLICATE = False
		if DEDUPLICATE:
			duplicates = get_duplicate_politicians(cur)
			print(duplicates)
			groupby_duplicates = create_name2ids_dict(duplicates)
			ignored_names = set([])


			for name, ids in groupby_duplicates.items():
				if name in ignored_names:
					continue
				check_and_merge_duplicates(name, ids, cur)
				conn.commit()
			

			


	print("Done.")

except Exception as e:
	if conn:
		conn.rollback()
	raise
finally:
	if conn:
		conn.close()

Connected.
Skipping 逢沢一郎 あいさわいちろう
Skipping 青木ひとみ あおきひとみ
Skipping 青柳仁士 あおやぎひとし
Skipping 青山繁晴 あおやましげはる
Skipping 青山周平 あおやましゅうへい
Skipping 赤澤亮正 あかざわりょうせい
Skipping 赤羽一嘉 あかばかずよし
Skipping あかま二郎 あかまじろう
Skipping 秋葉賢也 あきばけんや
Skipping 浅田眞澄美 あさだますみ
Skipping 浅野哲 あさのさとし
Skipping 東国幹 あずまくによし
Skipping 東徹 あずまとおる
Skipping 畦元将吾 あぜもとしょうご
Skipping 麻生太郎 あそうたろう
Skipping 阿部圭史 あべけいし
Skipping 阿部司 あべつかさ
Skipping あべ俊子 あべとしこ
Skipping 阿部弘樹 あべひろき
Skipping 有田芳生 ありたよしふ
Skipping 安藤たかお あんどうたかお
Skipping 飯泉嘉門 いいずみかもん
Skipping 五十嵐清 いがらしきよし
Skipping 池下卓 いけしたたく
Skipping 池畑浩太朗 いけはたこうたろう
Skipping 伊佐進一 いさしんいち
Skipping 石井啓一 いしいけいいち
Skipping 石井拓 いしいたく
Skipping 石川昭政 いしかわあきまさ
Skipping 石川勝 いしかわまさる
Skipping 石坂太 いしざかまさる
Skipping 石田真敏 いしだまさとし
Skipping 石破茂 いしばしげる
Skipping 石橋林太郎 いしばしりんたろう
Skipping 石原宏高 いしはらひろたか
Skipping 石原正敬 いしはらまさたか
Skipping 泉健太 いずみけんた
Skipping 一谷勇一郎 いちたにゆういちろう
Skipping 市村浩一郎 いちむらこういちろう
Skipping 井出庸生 いでようせい
Skipping 井戸まさえ いどまさえ
Skipping 伊藤恵介 いとうけいすけ
Skipping 伊藤聡 いとうさとし
Skipping 伊藤信太郎 いとうしんたろう
Skipping 伊藤忠彦 いとうただひこ
Skippi

Exception: Stop here